# Induction heads across a gap

**Scaffold only.** Ethan writes and dates the hypothesis before viewing held-out output, then runs and interprets this himself. Narrow replication of a known induction-head observation on GPT-2-small; gap length is one independent extension. [Olsson et al.](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) | [TransformerLens demo](https://transformerlensorg.github.io/TransformerLens/generated/demos/Main_Demo.html).

Set a Colab T4 GPU runtime if available. Run cells in order. No training.

## Pre-registered prediction - fill in before held-out runs

**Predicted direction and shape:** [Ethan writes here]

**Metric and disconfirming result:** [Ethan writes here]

**Date and saved version:** [Ethan writes here]

Keep this cell unchanged after seeing results. Write later discoveries in a dated post-hoc section.

In [ ]:
%pip -q install transformer-lens==4.0.0 numpy==2.2.6 matplotlib==3.10.6

In [ ]:
import json, platform, importlib.metadata as meta
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformer_lens import HookedTransformer
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.6.0"), "Colab torch >=2.6 required; restart runtime or choose a current runtime"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)
print("Device:", DEVICE, "PyTorch:", torch.__version__, "TransformerLens:", meta.version("transformer-lens"))
model = HookedTransformer.from_pretrained("gpt2-small", device=DEVICE)
model.eval()
assert model.cfg.n_ctx >= 328
print(model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_vocab, model.cfg.n_ctx)

## Fixed design

One shared 32-token target segment A per sample across gaps. The repeat always starts at absolute position 296. The variable random prefix (length `8 + 256 - gap`) holds the repeat position fixed while the source moves. We score `j = 0..30`: after seeing repeat `A[j]`, predict `A[j+1]`; attention target is the *first* `A[j+1]` at source index `prefix_len+j+1`. Tokens are sampled without replacement within each example to make accidental identical-token shortcuts rare. No model training.

In [ ]:
GAPS = (16, 64, 128, 256)
SEGMENT_LEN = 32
MAX_GAP = max(GAPS)
REPEAT_START = 8 + MAX_GAP + SEGMENT_LEN
DISCOVERY_SEED = 7301
DISCOVERY_N = 16
SEEDS = (11, 23, 37)  # held-out from discovery
N_PER_SEED = 16
BATCH_SIZE = 4  # lower to 2 for Colab out-of-memory
TOKEN_LOW, TOKEN_HIGH = 1000, 10000
assert DISCOVERY_SEED not in SEEDS
assert TOKEN_HIGH < model.cfg.d_vocab
assert REPEAT_START + SEGMENT_LEN <= model.cfg.n_ctx

def example(seed, sample_id, gap, nonrepeat=False):
    assert gap in GAPS
    # Independent deterministic stream per (seed, sample_id); paired across gaps.
    rng = np.random.default_rng(np.random.SeedSequence([seed, sample_id]))
    # One permutation supplies unique target, alternate source, foil and filler.
    pool = rng.permutation(np.arange(TOKEN_LOW, TOKEN_HIGH, dtype=np.int64))
    a = pool[:SEGMENT_LEN].copy()
    alternate = pool[SEGMENT_LEN:2*SEGMENT_LEN].copy()
    foil = int(pool[2*SEGMENT_LEN])
    prefix_len = 8 + MAX_GAP - gap
    prefix = pool[2*SEGMENT_LEN+1:2*SEGMENT_LEN+1+prefix_len]
    filler = pool[2*SEGMENT_LEN+1+prefix_len:2*SEGMENT_LEN+1+prefix_len+gap]
    source = alternate if nonrepeat else a
    tokens = np.concatenate([prefix, source, filler, a]).astype(np.int64)
    assert len(tokens) == REPEAT_START + SEGMENT_LEN
    source_positions = np.arange(prefix_len+1, prefix_len+SEGMENT_LEN)
    query_positions = np.arange(REPEAT_START, REPEAT_START+SEGMENT_LEN-1)
    assert np.array_equal(tokens[query_positions+1], a[1:])
    if not nonrepeat: assert np.array_equal(tokens[source_positions], a[1:])
    return tokens, source_positions, query_positions, a[1:], foil

def make_batch(seed, ids, gap, nonrepeat=False):
    examples = [example(seed, int(i), gap, nonrepeat) for i in ids]
    tokens = torch.as_tensor(np.stack([e[0] for e in examples]), dtype=torch.long, device=DEVICE)
    source = torch.as_tensor(np.stack([e[1] for e in examples]), dtype=torch.long, device=DEVICE)
    queries = torch.as_tensor(np.stack([e[2] for e in examples]), dtype=torch.long, device=DEVICE)
    targets = torch.as_tensor(np.stack([e[3] for e in examples]), dtype=torch.long, device=DEVICE)
    foils = torch.as_tensor([e[4] for e in examples], dtype=torch.long, device=DEVICE)
    return tokens, source, queries, targets, foils

# Index and pairing checks before running the model.
for gap in GAPS:
    ex = example(11, 0, gap)
    assert len(ex[0]) == 328 and ex[2][0] == REPEAT_START
    assert np.array_equal(ex[3], example(11, 0, GAPS[0])[3])
    assert ex[4] == example(11, 0, GAPS[0])[4]
print('Index/pairing checks passed')

## Discovery: attention-to-source (discovery set only)

Average attention paid to the earlier `A[j+1]` for each head. The model's hook exposes `[batch, head, destination, source]`. Scores are selected on **separate** samples from the test seeds, at the shortest gap only. This is a heuristic for choosing a candidate, not evidence of causality.

In [ ]:
def discover_heads():
    total = np.zeros((model.cfg.n_layers, model.cfg.n_heads), dtype=np.float64)
    count = 0
    for lo in range(0, DISCOVERY_N, BATCH_SIZE):
        ids = range(lo, min(lo+BATCH_SIZE, DISCOVERY_N))
        tok, src, query, target, foil = make_batch(DISCOVERY_SEED, ids, GAPS[0])
        bsz = tok.shape[0]
        batch_index = torch.arange(bsz, device=DEVICE)[:, None]
        def store_pattern(pattern, hook):
            # Use advanced indexing on batch and paired query/source axes.
            selected = pattern[batch_index, :, query, src]  # [batch, position, head]
            total[hook.layer()] += selected.detach().sum(dim=(0, 1)).cpu().numpy()
        model.run_with_hooks(tok, return_type=None, fwd_hooks=[(lambda name: name.endswith('attn.hook_pattern'), store_pattern)])
        count += bsz * query.shape[1]
        del tok
    scores = total / count
    return scores

scores = discover_heads()
ranking = np.argsort(scores.ravel())[::-1]
SELECTED_HEAD = tuple(map(int, np.unravel_index(ranking[0], scores.shape)))
CONTROL_HEAD = tuple(map(int, np.unravel_index(ranking[len(ranking)//2], scores.shape)))
assert SELECTED_HEAD != CONTROL_HEAD
print('Selected (layer, head):', SELECTED_HEAD, 'score:', scores[SELECTED_HEAD])
print('Control (layer, head):', CONTROL_HEAD, 'score:', scores[CONTROL_HEAD])
import csv
with open('head_scores.csv', 'w') as f:
    w = csv.writer(f); w.writerow(['layer', 'head', 'discovery_attention_to_source'])
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads): w.writerow([layer, head, float(scores[layer,head])])

## Held-out intervention

`hook_z` is the per-head output before the output projection; zeroing one head's z is a head-output intervention. Logit advantage is correct next token minus a fixed foil, averaged over 31 repeat positions in each prompt. We also collect mean attention to the designated source in the intact repeated prompt; attention is not itself a causal metric. Non-repeat replaces the first A with different tokens while leaving the repeat and target unchanged.

In [ ]:
def logits_and_attention(tokens, source, query, targets, foils, ablate=None, attention_head=None):
    bsz = tokens.shape[0]
    batch_idx = torch.arange(bsz, device=DEVICE)[:, None]
    attention = []
    hooks = []
    if attention_head is not None:
        layer, head = attention_head
        def read_pattern(pattern, hook):
            attention.append(pattern[batch_idx, head, query, source].detach().mean(dim=1).cpu().numpy())
        hooks.append((f'blocks.{layer}.attn.hook_pattern', read_pattern))
    if ablate is not None:
        layer, head = ablate
        def zero_head(z, hook):
            # clone rather than mutate hook inputs in place
            z = z.clone(); z[:, :, head, :] = 0
            return z
        hooks.append((f'blocks.{layer}.attn.hook_z', zero_head))
    with torch.inference_mode():
        logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
        by_pos = logits[batch_idx, query]  # [batch, scored_position, vocab]
        correct = by_pos.gather(-1, targets[..., None]).squeeze(-1)
        foil = by_pos.gather(-1, foils[:, None, None].expand(-1, query.shape[1], 1)).squeeze(-1)
        advantage = (correct - foil).mean(dim=1).detach().cpu().numpy()
    return advantage, (attention[0] if attention else None)

rows = []
for seed in SEEDS:
    for gap in GAPS:
        for lo in range(0, N_PER_SEED, BATCH_SIZE):
            ids = list(range(lo, min(lo+BATCH_SIZE, N_PER_SEED)))
            batch = make_batch(seed, ids, gap)
            clean, attn = logits_and_attention(*batch, attention_head=SELECTED_HEAD)
            ablated, _ = logits_and_attention(*batch, ablate=SELECTED_HEAD)
            control, _ = logits_and_attention(*batch, ablate=CONTROL_HEAD)
            nonrep = make_batch(seed, ids, gap, nonrepeat=True)
            nr_clean, _ = logits_and_attention(*nonrep)
            nr_ablated, _ = logits_and_attention(*nonrep, ablate=SELECTED_HEAD)
            for k, sample_id in enumerate(ids):
                rows.append(dict(seed=seed, sample_id=sample_id, gap=gap,
                    attention_to_source=float(attn[k]), clean_advantage=float(clean[k]),
                    ablated_advantage=float(ablated[k]), control_advantage=float(control[k]),
                    nonrepeat_clean_advantage=float(nr_clean[k]),
                    nonrepeat_ablated_advantage=float(nr_ablated[k]),
                    selected_effect=float(clean[k]-ablated[k]),
                    control_effect=float(clean[k]-control[k]),
                    nonrepeat_selected_effect=float(nr_clean[k]-nr_ablated[k])))
        print('Finished held-out seed/gap:', seed, gap)
with open('results.csv','w',newline='') as f:
    writer=csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
print('Saved', len(rows), 'prompt-level rows; no duplicate discovery seeds')

## Descriptive summaries and plot

Bootstrap resamples the held-out *sample IDs* within seed, with the same IDs across gaps to preserve pairing. A small number of seeds and one fixed model do not support a population-level generalization claim. Inspect individual results and control contrasts, not only this figure.

In [ ]:
def summarize_paired(rows, key, n_boot=1000):
    rng = np.random.default_rng(941)
    seeds = sorted({r['seed'] for r in rows})
    by_seed = {seed: {gap: np.array([r[key] for r in rows if r['seed']==seed and r['gap']==gap])
                      for gap in GAPS} for seed in seeds}
    for seed in seeds:
        assert all(len(by_seed[seed][gap]) == N_PER_SEED for gap in GAPS)
    means, lows, highs = [], [], []
    for gap in GAPS:
        values = np.concatenate([by_seed[seed][gap] for seed in seeds])
        draws = []
        for _ in range(n_boot):
            resampled = []
            for seed in seeds:
                indices = rng.integers(0, N_PER_SEED, N_PER_SEED)
                resampled.extend(by_seed[seed][gap][indices])
            draws.append(np.mean(resampled))
        means.append(float(np.mean(values)))
        lows.append(float(np.quantile(draws,.025))); highs.append(float(np.quantile(draws,.975)))
    return np.array(means), np.array(lows), np.array(highs)

fig, axes=plt.subplots(1, 2, figsize=(10, 3.5))
for key, label in [('selected_effect','Selected head'), ('control_effect','Control head'),
                   ('nonrepeat_selected_effect','Selected head, no repeat')]:
    mean, lo, hi=summarize_paired(rows,key)
    axes[0].plot(GAPS,mean,marker='o',label=label)
    axes[0].fill_between(GAPS,lo,hi,alpha=.12)
mean,lo,hi=summarize_paired(rows,'attention_to_source')
axes[1].plot(GAPS,mean,marker='o',color='teal')
axes[1].fill_between(GAPS,lo,hi,alpha=.18,color='teal')
axes[0].axhline(0,color='gray',lw=.7); axes[0].set_ylabel('Clean minus ablated logit advantage')
axes[1].set_ylabel('Mean attention to earlier next-token source')
for ax in axes: ax.set_xlabel('Filler gap (tokens)'); ax.set_xticks(GAPS)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig('gap_plot.png',dpi=160); plt.show()
print('Effect means:', summarize_paired(rows,'selected_effect')[0])
print('Control means:', summarize_paired(rows,'control_effect')[0])

In [ ]:
metadata = dict(model='gpt2-small', selected_head=SELECTED_HEAD, control_head=CONTROL_HEAD,
                discovery_seed=DISCOVERY_SEED, discovery_n=DISCOVERY_N, heldout_seeds=SEEDS,
                n_per_seed=N_PER_SEED, gaps=GAPS, segment_len=SEGMENT_LEN,
                device=DEVICE, python=platform.python_version(), torch=torch.__version__,
                transformer_lens=meta.version('transformer-lens'), numpy=np.__version__,
                matplotlib=meta.version('matplotlib'))
with open('run_metadata.json','w') as f: json.dump(metadata,f,indent=2)
print('Saved metadata; download all generated files from the Colab Files pane.')

## Ethan's analysis - deliberately blank

**Observations, including nulls:** [write after execution]

**Control contrasts and uncertainty:** [write after execution]

**Limitations / alternate explanations:** [write after execution]

**Next experiment and prior work check:** [write after execution]

Do not call this a reproduction of Olsson et al.'s training-time phase change or a claim about all in-context learning.